In [ ]:
# ============================================================
# fix_scanned_pdfs.ipynb
#
# PURPOSE: Fix the 12 conf=0.00 records where pdfplumber
# failed because the PDF is image-based (scanned).
#
# APPROACH: Convert PDF pages to images -> send to GPT-4o vision
# GPT-4o reads the image like a human - works on any layout.
#
# WHICH PDFs: Only the failed records identified in analysis.
#
# PAGES SENT: pages 1-6 (indices 0-5) as images
# ============================================================

In [1]:
# CELL 1 - Imports
import subprocess
subprocess.run(['pip', 'install', 'pymupdf', 'openai',
                'psycopg2-binary', 'python-dotenv',
                '--quiet'], check=False)

import fitz
from openai import OpenAI
import psycopg2
from pathlib import Path
from dotenv import load_dotenv
import os, re, json, time, base64
from io import BytesIO

load_dotenv()

PDF_BASE_DIR   = Path(r'C:\\Users\\Negza\\Desktop\\projects\\pfe\\bvmt_project\\data\\financials')
GPT_MODEL      = 'gpt-4o'
PAGES_TO_SEND  = 7
IMAGE_DPI      = 150

GITHUB_TOKENS = [
    t for t in [
        os.getenv('GITHUB_TOKEN_1'),
        os.getenv('GITHUB_TOKEN_2'),
        os.getenv('GITHUB_TOKEN_3'),
        os.getenv('GITHUB_TOKEN_4'),
    ] if t
]
REQUESTS_PER_TOKEN = 47
_token_index       = 0
_token_requests    = [0] * len(GITHUB_TOKENS)

def get_client():
    return OpenAI(
        base_url='https://models.inference.ai.azure.com',
        api_key=GITHUB_TOKENS[_token_index],
    )

def get_conn():
    return psycopg2.connect(
        host=os.getenv('DB_HOST'),
        port=int(os.getenv('DB_PORT', 5432)),
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD')
    )

_r = get_client().chat.completions.create(
    model=GPT_MODEL,
    messages=[{'role': 'user', 'content': 'Reply: READY'}],
    max_tokens=5
)
_token_requests[_token_index] += 1
print(f'GPT-4o vision: {_r.choices[0].message.content.strip()}')
print(f'Tokens loaded: {len(GITHUB_TOKENS)}')
print('Cell 1 OK')

GPT-4o vision: READY! How can I
Tokens loaded: 4
Cell 1 OK


In [2]:
# CELL 2 - PDF to images converter
def pdf_pages_to_base64(pdf_path: Path,
                        page_indices: list,
                        dpi: int = 150) -> list:
    """Convert specific PDF pages to base64 PNG strings."""
    images_b64 = []
    try:
        doc = fitz.open(str(pdf_path))
        mat = fitz.Matrix(dpi / 72, dpi / 72)

        for idx in sorted(set(page_indices)):
            if 0 <= idx < len(doc):
                pix  = doc[idx].get_pixmap(matrix=mat)
                png  = pix.tobytes('png')
                b64  = base64.b64encode(png).decode('utf-8')
                images_b64.append(b64)

        doc.close()
    except Exception as e:
        print(f'  Image conversion error: {e}')

    return images_b64

print('Cell 2 OK - pdf_pages_to_base64 defined')

Cell 2 OK - pdf_pages_to_base64 defined


In [9]:
# CELL 3 - Vision prompts
SYSTEM_VISION = """You are a precise financial data extraction assistant.
You are looking at scanned images of Tunisian company annual report pages.
Extract specific financial figures and return valid JSON only.
No markdown, no explanation, no code fences - only the JSON object."""

PROMPT_VISION_BANK = """You are looking at scanned images of a TUNISIAN BANK annual report.

UNIT: kDT (thousands of Tunisian Dinars). Return values AS-IS - do NOT divide.

Read these pages carefully and find:

FROM BALANCE SHEET (Bilan):
- total_assets: grand total 'Total des actifs' or 'TOTAL ACTIF'
- total_liabilities: 'Total des passifs'
- total_loans: 'Creances sur la clientele' (row AC3)
- total_deposits: 'Depots et avoirs de la clientele' (row PA3)
- equity: 'Total des capitaux propres' (positive number, 300k-3M range)

FROM P&L (Etat de Resultat):
- pnb: 'Produit Net Bancaire'
- revenue: null
- net_result: 'Resultat net de l'exercice'
- operating_expenses: 'Charges operatoires' (positive number)

FROM NOTES (if visible):
- npl_ratio: 'Taux des engagements classes' percentage e.g. 9.96
- coverage_ratio: 'Taux de couverture' percentage e.g. 75.18
- lcr: LCR percentage e.g. 142.97

RULES:
1. Return ONLY valid JSON - no markdown, no text before or after
2. Use the FIRST/MOST RECENT value column
3. Use null for any value not clearly visible
4. Remove spaces from numbers: 12 348 844 -> 12348844
5. Negative values: -4401 not (4401)

Return exactly:
{"total_assets":null,"total_liabilities":null,"total_loans":null,"total_deposits":null,"equity":null,"pnb":null,"revenue":null,"net_result":null,"operating_expenses":null,"npl_ratio":null,"coverage_ratio":null,"lcr":null}"""

# Use a plain token and replace it later to avoid .format() conflicts with JSON braces.
PROMPT_VISION_DT = """You are looking at scanned images of a Tunisian __COMPANY_TYPE__ annual report.

UNIT: FULL DINARS (DT). Divide ALL monetary values by 1000.
Example: 272 682 126 -> return 272682.126
Example: 40 416 678  -> return 40416.678
Percentages (npl_ratio, coverage_ratio, lcr) are NOT divided.

Read these pages carefully and find:

FROM BALANCE SHEET (Bilan):
- total_assets: grand total only - NOT subtotals like 'Total actifs immobilises'
- total_liabilities: 'Total des passifs' grand total
- total_loans: null (non-bank)
- total_deposits: null (non-bank)
- equity: 'Total des capitaux propres'

FROM P&L:
- pnb: null (non-bank)
- revenue: 'Chiffre d'affaires' or 'Revenus' or 'Produits des loyers' (divide by 1000)
- net_result: 'Resultat net de l'exercice' or 'Resultat de l'exercice' (divide by 1000)
- operating_expenses: total charges (positive, divide by 1000)

RULES:
1. Return ONLY valid JSON - no markdown, no text
2. Divide ALL monetary values by 1000
3. Use null for any value not clearly visible
4. Use the MOST RECENT column (leftmost or current year)

Return exactly:
{"total_assets":null,"total_liabilities":null,"total_loans":null,"total_deposits":null,"equity":null,"pnb":null,"revenue":null,"net_result":null,"operating_expenses":null,"npl_ratio":null,"coverage_ratio":null,"lcr":null}"""

print('Cell 3 OK - vision prompts defined')

Cell 3 OK - vision prompts defined


In [10]:
# CELL 4 - Vision API call
COMPANY_TYPE_MAP = {
    'AMEN BANK':'bank','ATB':'bank','ATTIJARI BANK':'bank',
    'BIAT':'bank','BNA':'bank','BH':'bank','STB':'bank',
    'UIB':'bank','UBCI':'bank',
    'ATL':'leasing','ATTIJARI LEASING':'leasing','CIL':'leasing',
    'HANNIBAL LEASE':'leasing','MODERN LEASING':'leasing',
    'ASSUR MAGHREBIA':'insurance','ASSU MAGHREBIA VIE':'insurance',
    'BNA ASSURANCES':'insurance','ICF':'insurance',
}
ALWAYS_DT = {
    'ATL','ATTIJARI LEASING','CIL','HANNIBAL LEASE','MODERN LEASING',
    'ASSUR MAGHREBIA','ASSU MAGHREBIA VIE','BNA ASSURANCES','ICF',
    'ARTES','SFBT','CEREALIS','CARTHAGE CEMENT','CIMENTS DE BIZERTE',
    'ESSOUKNA','SOPAT','NEW BODY LINE','CELLCOM','EURO-CYCLES',
    'SOTETEL','TUNISIE VALEURS','SAH','SOTUMAG','SERVICOM',
    'ONE TECH HOLDING','ADWYA',
}

def get_company_type(ticker):
    return COMPANY_TYPE_MAP.get(ticker, 'non_bank')

def get_unit(ticker):
    if ticker in ALWAYS_DT:
        return 'DT'
    ctype = get_company_type(ticker)
    return 'kDT' if ctype == 'bank' else 'DT'

def _rotate_token() -> bool:
    global _token_index
    if _token_index + 1 >= len(GITHUB_TOKENS):
        return False
    _token_index += 1
    print(f'\n  [Token -> account {_token_index+1}/{len(GITHUB_TOKENS)}]', end='', flush=True)
    return True

def call_vision(images_b64: list, ticker: str, retries: int = 3) -> tuple:
    global _token_requests

    if not images_b64:
        return {}, 'no_images'

    company_type = get_company_type(ticker)
    unit = get_unit(ticker)

    if company_type == 'bank':
        text_prompt = PROMPT_VISION_BANK
    else:
        text_prompt = PROMPT_VISION_DT.replace('__COMPANY_TYPE__', company_type)

    content = [{"type": "text", "text": text_prompt}]
    for b64 in images_b64:
        content.append({
            'type': 'image_url',
            'image_url': {
                'url': f'data:image/png;base64,{b64}',
                'detail': 'high'
            }
        })

    for attempt in range(1, retries + 1):
        if _token_requests[_token_index] >= REQUESTS_PER_TOKEN:
            if not _rotate_token():
                return {}, 'all_tokens_exhausted'

        try:
            client = get_client()
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_VISION},
                    {'role': 'user', 'content': content}
                ],
                max_tokens=400,
                temperature=0.0,
            )
            _token_requests[_token_index] += 1

            raw = response.choices[0].message.content.strip()
            raw = re.sub(r'^```(?:json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw)
            data = json.loads(raw.strip())
            return data, 'OK'

        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                if not _rotate_token():
                    return {}, 'all_tokens_exhausted'
                time.sleep(3)
            elif '500' in err:
                if attempt < retries:
                    time.sleep(15)
                else:
                    return {}, 'server_error'
            elif 'json' in err.lower():
                return {}, 'json_parse_error'
            else:
                if attempt < retries:
                    time.sleep(8)
                else:
                    return {}, f'error:{err[:80]}'

    return {}, 'max_retries_exceeded'

print('Cell 4 OK - call_vision defined')

Cell 4 OK - call_vision defined


In [5]:
# CELL 5 - Validation
MAIN_FIELDS_BANK    = ['total_assets','total_liabilities','total_loans',
                       'total_deposits','equity','pnb','net_result',
                       'operating_expenses']
MAIN_FIELDS_NONBANK = ['total_assets','total_liabilities','equity',
                       'revenue','net_result']

def validate_extraction(data: dict, company_type: str) -> tuple:
    notes = []
    needs_review = False
    valid_count = 0

    max_kdt = 30_000_000 if company_type == 'bank' else 10_000_000
    min_kdt = 10
    main_fields = MAIN_FIELDS_BANK if company_type == 'bank' else MAIN_FIELDS_NONBANK

    for field in main_fields + ['operating_expenses']:
        val = data.get(field)
        if val is None:
            continue
        if not isinstance(val, (int, float)):
            data[field] = None
            notes.append(f'{field}:not_numeric')
            continue
        if abs(val) > 0:
            if abs(val) < min_kdt:
                data[field] = None
                notes.append(f'{field}:too_small({val:.1f})')
                continue
            if abs(val) > max_kdt:
                corrected = val / 1000
                if min_kdt <= abs(corrected) <= max_kdt:
                    data[field] = round(corrected, 3)
                    notes.append(f'{field}:auto_div1000')
                    val = corrected
                else:
                    data[field] = None
                    notes.append(f'{field}:out_of_range')
                    continue
        if field in main_fields and data.get(field) is not None:
            valid_count += 1

    for field in ['npl_ratio', 'coverage_ratio', 'lcr']:
        val = data.get(field)
        if val is None:
            continue
        if not isinstance(val, (int, float)) or not (0 < val < 300):
            data[field] = None
            notes.append(f'{field}:invalid')

    ta = data.get('total_assets')
    nr = data.get('net_result')
    tl = data.get('total_liabilities')
    eq = data.get('equity')

    if ta and nr and abs(nr) > abs(ta) * 0.5:
        needs_review = True
        notes.append('net_gt_50pct_assets')

    if ta and tl and eq and ta > 0:
        diff = abs(ta - (tl + eq)) / ta
        tol = 0.05 if company_type == 'bank' else 0.30
        if diff > tol:
            needs_review = True
            notes.append(f'balance_diff:{diff*100:.1f}%')

    if not data.get('total_assets'):
        needs_review = True
        notes.append('total_assets:missing')

    if company_type != 'bank':
        data['pnb'] = None

    confidence = round(min(valid_count / len(main_fields), 1.0), 3)
    return data, confidence, needs_review, (' | '.join(notes) or 'OK')

print('Cell 5 OK - validate_extraction defined')

Cell 5 OK - validate_extraction defined


In [6]:
# CELL 6 - DB helpers
def get_isin_map() -> dict:
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('SELECT ticker, isin_code FROM company_metadata')
            return {r[0]: r[1] for r in cur.fetchall()}
    finally:
        conn.close()

def compute_derived(data: dict) -> dict:
    net = data.get('net_result')
    eq  = data.get('equity')
    if net and eq and eq > 0:
        data['roe'] = round((net / eq) * 100, 4)
    pnl  = data.get('pnb') or data.get('revenue')
    opex = data.get('operating_expenses')
    if opex and pnl and pnl > 0:
        data['cost_income_ratio'] = round(abs(opex) / pnl * 100, 4)
    loans = data.get('total_loans')
    dep   = data.get('total_deposits')
    if loans and dep and dep > 0:
        data['loan_to_deposit'] = round(loans / dep * 100, 4)
    return data

def upsert_record(ticker, isin, period, period_end_date,
                  source_pdf, company_type, data,
                  confidence, needs_review, notes):
    data = compute_derived(data.copy())
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('''
                INSERT INTO financial_statements (
                    ticker, isin_code, period, period_end_date, source_pdf,
                    company_type,
                    total_assets, total_liabilities, total_loans,
                    total_deposits, equity,
                    pnb, revenue, net_result, operating_expenses,
                    npl_ratio, coverage_ratio, lcr,
                    roe, cost_income_ratio, loan_to_deposit,
                    extraction_confidence, needs_review, extraction_notes
                ) VALUES (
                    %s,%s,%s,%s,%s,%s,
                    %s,%s,%s,%s,%s,
                    %s,%s,%s,%s,
                    %s,%s,%s,
                    %s,%s,%s,
                    %s,%s,%s
                )
                ON CONFLICT (ticker, period) DO UPDATE SET
                    total_assets          = EXCLUDED.total_assets,
                    total_liabilities     = EXCLUDED.total_liabilities,
                    total_loans           = EXCLUDED.total_loans,
                    total_deposits        = EXCLUDED.total_deposits,
                    equity                = EXCLUDED.equity,
                    pnb                   = EXCLUDED.pnb,
                    revenue               = EXCLUDED.revenue,
                    net_result            = EXCLUDED.net_result,
                    operating_expenses    = EXCLUDED.operating_expenses,
                    npl_ratio             = EXCLUDED.npl_ratio,
                    coverage_ratio        = EXCLUDED.coverage_ratio,
                    lcr                   = EXCLUDED.lcr,
                    roe                   = EXCLUDED.roe,
                    cost_income_ratio     = EXCLUDED.cost_income_ratio,
                    loan_to_deposit       = EXCLUDED.loan_to_deposit,
                    extraction_confidence = EXCLUDED.extraction_confidence,
                    needs_review          = EXCLUDED.needs_review,
                    extraction_notes      = EXCLUDED.extraction_notes,
                    scraped_at            = NOW()
            ''', (
                ticker, isin, period, period_end_date,
                source_pdf, company_type,
                data.get('total_assets'), data.get('total_liabilities'),
                data.get('total_loans'), data.get('total_deposits'),
                data.get('equity'),
                data.get('pnb'), data.get('revenue'),
                data.get('net_result'), data.get('operating_expenses'),
                data.get('npl_ratio'), data.get('coverage_ratio'),
                data.get('lcr'),
                data.get('roe'), data.get('cost_income_ratio'),
                data.get('loan_to_deposit'),
                confidence, needs_review, notes
            ))
        conn.commit()
    except Exception as e:
        conn.rollback()
        print(f'\n  DB error {ticker} {period}: {e}')
    finally:
        conn.close()

print('Cell 6 OK - DB helpers defined')

Cell 6 OK - DB helpers defined


In [11]:
# CELL 7 - Fix the failed PDFs
FAILED_RECORDS = [
    ('ADWYA',              'FY 2022', '2022-12-31', 'adwya_efd311222.pdf'),
    ('ASSU MAGHREBIA VIE', 'FY 2018', '2018-12-31', 'maghrebia_vie_efd311218.pdf'),
    ('CELLCOM',            'FY 2019', '2019-12-31', 'cellcom_efd311219.pdf'),
    ('CELLCOM',            'FY 2020', '2020-12-31', 'cellcom_efd311220.pdf'),
    ('ICF',                'FY 2018', '2018-12-31', 'icf_efd_2018.pdf'),
    ('ONE TECH HOLDING',   'FY 2018', '2018-12-31', 'oth_efd311218.pdf'),
    ('ONE TECH HOLDING',   'FY 2020', '2020-12-31', 'oth_efd311220.pdf'),
    ('ONE TECH HOLDING',   'FY 2021', '2021-12-31', 'oth_efd311221.pdf'),
    ('ONE TECH HOLDING',   'FY 2022', '2022-12-31', 'oth_efd311222.pdf'),
    ('ONE TECH HOLDING',   'FY 2024', '2024-12-31', 'oth_efd311224.pdf'),
    ('SOPAT',              'FY 2022', '2022-12-31', 'sopat_efd_2022.pdf'),
    ('SOPAT',              'FY 2023', '2023-12-31', 'sopat_efd311223.pdf'),
    ('UIB',                'FY 2018', '2018-12-31', 'uib_efd311218.pdf'),
]

isin_map = get_isin_map()
total = len(FAILED_RECORDS)

print(f'Processing {total} failed PDFs with vision...')
print(f'Requests available: {len(GITHUB_TOKENS) * REQUESTS_PER_TOKEN - sum(_token_requests)}')
print()

fixed = 0
still_failed = 0

for i, (ticker, period, ped, filename) in enumerate(FAILED_RECORDS, 1):
 # Build PDF path
    safe     = re.sub(r'[<>:"/\\|?*]', '_', ticker)
    pdf_path = PDF_BASE_DIR / safe / 'FY_ANNUAL' / filename
 
    if not pdf_path.exists():
        print(f"[{i:2d}/{total}] ✗ NOT FOUND: {ticker} {period} — {pdf_path}")
        still_failed += 1
        continue
 
    company_type = get_company_type(ticker)
    unit         = get_unit(ticker)
 
    print(f"[{i:2d}/{total}] {ticker:<25} {period}  "
          f"type={company_type}  unit={unit}",
          end=' ', flush=True)
 
    # Convert pages 1-6 (indices 0-5) to base64 images
    page_indices = list(range(PAGES_TO_SEND))  # [0,1,2,3,4,5]
    images_b64   = pdf_pages_to_base64(pdf_path, page_indices, dpi=IMAGE_DPI)
 
    if not images_b64:
        print(f"✗ image_conversion_failed")
        upsert_record(ticker, isin_map.get(ticker), period, ped,
                      filename, company_type, {}, 0.0, True,
                      'vision:image_conversion_failed')
        still_failed += 1
        continue
 
    print(f"({len(images_b64)} pages) ", end='', flush=True)
 
    # Call GPT-4o vision
    data, api_note = call_vision(images_b64, ticker)
 
    if api_note == 'all_tokens_exhausted':
        print(f"\n\nAll tokens exhausted — stopping.")
        print(f"Run again tomorrow for remaining records.")
        break
 
    if not data:
        print(f"✗ {api_note}")
        upsert_record(ticker, isin_map.get(ticker), period, ped,
                      filename, company_type, {}, 0.0, True,
                      f'vision_failed:{api_note}')
        still_failed += 1
        continue
 
    # Validate
    data, confidence, needs_review, val_notes = validate_extraction(
        data, company_type)
 
    final_notes = f'vision | {val_notes}'
 
    # Upsert into DB
    upsert_record(ticker, isin_map.get(ticker), period, ped,
                  filename, company_type, data,
                  confidence, needs_review, final_notes)
 
    icon = '✓' if not needs_review else '⚠'
    print(f"{icon} conf={confidence:.2f}  {val_notes[:50]}")
 
    if confidence >= 0.5:
        fixed += 1
    else:
        still_failed += 1
 
    time.sleep(2)
 
print()
print("=" * 60)
print(f"DONE — Fixed: {fixed}/{total}  Still failed: {still_failed}/{total}")
print(f"Requests used: {sum(_token_requests)}")
print("=" * 60)
 
 

Processing 13 failed PDFs with vision...
Requests available: 187

[ 1/13] ADWYA                     FY 2022  type=non_bank  unit=DT (7 pages) ✓ conf=1.00  OK
[ 2/13] ASSU MAGHREBIA VIE        FY 2018  type=insurance  unit=DT (7 pages) ✓ conf=1.00  OK
[ 3/13] CELLCOM                   FY 2019  type=non_bank  unit=DT (7 pages) ✓ conf=1.00  OK
[ 4/13] CELLCOM                   FY 2020  type=non_bank  unit=DT (7 pages) ✓ conf=1.00  OK
[ 5/13] ICF                       FY 2018  type=insurance  unit=DT (7 pages) ✓ conf=1.00  OK
[ 6/13] ONE TECH HOLDING          FY 2018  type=non_bank  unit=DT (7 pages) ✓ conf=1.00  OK
[ 7/13] ONE TECH HOLDING          FY 2020  type=non_bank  unit=DT (7 pages) ✓ conf=1.00  OK
[ 8/13] ONE TECH HOLDING          FY 2021  type=non_bank  unit=DT (7 pages) ✓ conf=1.00  OK
[ 9/13] ONE TECH HOLDING          FY 2022  type=non_bank  unit=DT (7 pages) ✓ conf=1.00  operating_expenses:too_small(5.6)
[10/13] ONE TECH HOLDING          FY 2024  type=non_bank  unit=DT (7 page

In [12]:
# CELL 8 - Verify results
import pandas as pd

conn = get_conn()
tickers_to_check = list(set(r[0] for r in FAILED_RECORDS))
placeholders = ','.join(['%s'] * len(tickers_to_check))

df = pd.read_sql(f'''
    SELECT ticker, period, company_type,
           total_assets, equity, net_result, pnb, revenue,
           extraction_confidence, needs_review, extraction_notes
    FROM financial_statements
    WHERE ticker IN ({placeholders})
      AND period LIKE 'FY%%'
    ORDER BY ticker, period
''', conn, params=tickers_to_check)
conn.close()

print('Results for previously failed records:')
print(df.to_string(index=False))

Results for previously failed records:
            ticker  period company_type  total_assets      equity  net_result      pnb    revenue  extraction_confidence  needs_review                           extraction_notes
             ADWYA FY 2016     non_bank     85735.875   34183.160    2734.405      NaN  84646.777                  1.000         False                                         OK
             ADWYA FY 2017     non_bank     90571.902   39032.619    4914.454      NaN  95337.301                  1.000         False                                         OK
             ADWYA FY 2018     non_bank    112132.832   39002.918    3122.741      NaN 108179.015                  1.000         False                                         OK
             ADWYA FY 2019     non_bank    114190.019   38336.184    1547.713      NaN 105638.078                  1.000         False                                         OK
             ADWYA FY 2020     non_bank    133747.104   39888.909    16

C:\Users\Negza\AppData\Local\Temp\ipykernel_6160\903421737.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f'''
